In [26]:
import requests
import json
import os
from datetime import datetime, timezone
import random
import asyncio, aiohttp
import aiolimiter
from tqdm.auto import tqdm
from tqdm.asyncio import tqdm_asyncio


In [28]:

def load_posts_dict(file_paths):
    """
    Takes a list of .jsonl file paths and returns a dict:

        {
            <source_name>: { post_uri -> post_data },
            ...
        }

    where source_name is derived from the filename (without extension).
    """

    all_posts = {}

    for file_path in file_paths:
        source_name = os.path.splitext(os.path.basename(file_path))[0]

        posts_dict = {}
        bad_lines = 0

        with open(file_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    post = json.loads(line)
                except json.JSONDecodeError:
                    bad_lines += 1
                    continue

                post_id = (
                    post.get("uri")
                    or post.get("post", {}).get("uri")
                    or post.get("id")
                )

                if not post_id:
                    continue

                # optional: annotate post with source
                post["_source"] = source_name

                posts_dict[post_id] = post

        all_posts[source_name] = posts_dict

        print(
            f"Loaded {len(posts_dict)} posts from {file_path} "
            f"({bad_lines} bad lines skipped)"
        )

    return all_posts


In [29]:
files = [
    "politics.jsonl",
    "sports.jsonl",
]

posts = load_posts_dict(files)


Loaded 100 posts from politics.jsonl (0 bad lines skipped)
Loaded 100 posts from sports.jsonl (0 bad lines skipped)


In [ ]:


REPOST_API  = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"
PROFILE_API = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
FEED_API    = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
FOLLOW_API  = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

HEADERS = {"User-Agent": "repost-prediction-research/1.0"}

# --------------------------------------------------
# UTILS
# --------------------------------------------------

def parse_dt(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)

def get_author_dids(posts_dict):
    return {
        p.get("author", {}).get("did")
        or p.get("post", {}).get("author", {}).get("did")
        for p in posts_dict.values()
        if p
    }

def flatten_posts_dict(posts_by_tag):
    """
    Flattens:
        { tag: { post_uri -> post_data } }
    into:
        { post_uri -> post_data }
    """
    flat_posts = {}

    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            flat_posts[uri] = post

    return flat_posts

# --------------------------------------------------
# FETCH REPOSTERS (AND ATTACH TO POSTS)
# --------------------------------------------------

async def fetch_reposters(session, uri):
    async with session.get(
        REPOST_API, params={"uri": uri, "limit": 100}, headers=HEADERS
    ) as r:
        if r.status != 200:
            return []
        data = await r.json()
        return [u["did"] for u in data.get("repostedBy", [])]

async def collect_all_reposters(posts_dict, concurrency=50):
    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    reposters = set()

    async with aiohttp.ClientSession(connector=connector) as session:

        async def fetch_with_uri(uri):
            reposted_by = await fetch_reposters(session, uri)
            return uri, reposted_by

        tasks = [
            fetch_with_uri(uri)
            for uri, p in posts_dict.items()
            if p.get("repostCount", 0) > 0
        ]

        for coro in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Reposters",
            unit="post"
        ):
            uri, reposted_by = await coro

            posts_dict[uri]["reposted_by"] = reposted_by
            reposters.update(reposted_by)

    return reposters

# --------------------------------------------------
# FETCH PROFILE / FOLLOWS / HISTORY
# --------------------------------------------------

async def fetch_profile(session, did):
    async with session.get(
        PROFILE_API, params={"actor": did}, headers=HEADERS
    ) as r:
        if r.status != 200:
            return None
        return await r.json()

async def fetch_followed_authors(session, did, author_set):
    async with session.get(
        FOLLOW_API,
        params={"actor": did, "limit": 100},
        headers=HEADERS
    ) as r:
        if r.status != 200:
            return []
        data = await r.json()
        return [
            u["did"] for u in data.get("follows", [])
            if u.get("did") in author_set
        ]

async def fetch_history(session, did, limit=50):
    history, cursor = [], None

    while len(history) < limit:
        params = {"actor": did, "limit": min(100, limit - len(history))}
        if cursor:
            params["cursor"] = cursor

        async with session.get(FEED_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                break

            data = await r.json()
            feed = data.get("feed", [])
            if not feed:
                break

            for item in feed:
                post = item.get("post")
                if not post:
                    continue

                record = post.get("record", {})
                reason = item.get("reason", {})

                #  REPOST (WITH TIME)
                if reason.get("$type", "").endswith("reasonRepost"):
                    history.append({
                        "type": "repost",
                        "post_uri": post["uri"],
                        "post_author_did": post["author"]["did"],
                        "reposted_at": reason.get("indexedAt"),
                    })
                    continue

                #  ORIGINAL POST (non-reply)
                if "reply" not in record:
                    history.append({
                        "type": "post",
                        "post_uri": post["uri"],
                        "created_at": record.get("createdAt"),
                        "text": record.get("text", ""),
                        "likes": post.get("likeCount"),
                        "reposts": post.get("repostCount"),
                        "replies": post.get("replyCount"),
                        "quotes": post.get("quoteCount"),
                    })

                if len(history) >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                break

    return history

# --------------------------------------------------
# MAIN BUILDER (FAST)
# --------------------------------------------------

async def build_users_from_posts(posts_dict, concurrency=300):
    author_set = get_author_dids(posts_dict)
    reposters = await collect_all_reposters(posts_dict)

    user_dids = reposters | author_set
    users = {}

    connector = aiohttp.TCPConnector(limit=0, limit_per_host=0, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10)

    follows_limiter = aiolimiter.AsyncLimiter(30, 1)
    sem = asyncio.Semaphore(concurrency)

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout
    ) as session:

        async def process(did):
            try:
                async with sem:
                    profile_task = fetch_profile(session, did)
                    history_task = fetch_history(session, did)

                    async with follows_limiter:
                        follows_task = fetch_followed_authors(
                            session, did, author_set
                        )

                    profile, history, follows = await asyncio.gather(
                        profile_task,
                        history_task,
                        follows_task,
                        return_exceptions=True
                    )

                if isinstance(profile, Exception) or not profile:
                    return

                if isinstance(history, Exception):
                    history = []

                if isinstance(follows, Exception):
                    follows = []

            except asyncio.TimeoutError:
                return
            except Exception as e:
                return

            post_uri_set = set(posts_dict.keys())
            reposts_of_posts = [
                {
                    "post_uri": h["post_uri"],
                    "reposted_at": h["reposted_at"],
                }
                for h in history
                if h.get("type") == "repost" and h["post_uri"] in post_uri_set
            ]

            created = profile.get("createdAt")
            age_days = (
                max(1, (datetime.now(timezone.utc) - parse_dt(created)).days)
                if created else None
            )

            users[did] = {
                "profile": {
                    "did": did,
                    "handle": profile.get("handle"),
                    "display_name": profile.get("displayName"),
                    "description": profile.get("description"),
                    "created_at": created,
                },
                "stats": {
                    "followers": profile.get("followersCount"),
                    "follows": profile.get("followsCount"),
                    "posts": profile.get("postsCount"),
                    "account_age_days": age_days,
                },
                "follows_authors": follows,
                "reposts_of_posts": reposts_of_posts,  
                "history": history,
            }

        tasks = [process(did) for did in user_dids]

        for t in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Building users",
            unit="user"
        ):
            await t

    return users


In [ ]:
post_dict = flatten_posts_dict(posts)
users = await build_users_from_posts(post_dict)

Building users: 100%|██████████| 314/314 [00:37<00:00,  8.43user/s]


In [32]:
with open("test.json", "w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)


In [7]:
def print_basic_stats(users, posts_dict):
    # -------------------------
    # Authors followed per user
    # -------------------------
    follow_counts = [
        len(u.get("follows_authors", []))
        for u in users.values()
    ]

    avg_authors_followed = (
        sum(follow_counts) / len(follow_counts)
        if follow_counts else 0
    )
    max_authors_followed = max(follow_counts) if follow_counts else 0

    # -------------------------
    # Reposts per post
    # -------------------------
    repost_counts = [
        len(p.get("reposted_by", []))
        for p in posts_dict.values()
    ]

    avg_reposts_per_post = (
        sum(repost_counts) / len(repost_counts)
        if repost_counts else 0
    )
    max_reposts_per_post = max(repost_counts) if repost_counts else 0

    posts_gt_5 = sum(c > 5 for c in repost_counts)
    pct_posts_gt_5 = (
        100 * posts_gt_5 / len(repost_counts)
        if repost_counts else 0
    )

    # -------------------------
    # Reposts per user
    # -------------------------
    reposts_per_user = [
        len(u.get("reposts_of_posts", []))
        for u in users.values()
    ]

    users_gt_5 = sum(c > 5 for c in reposts_per_user)
    pct_users_gt_5 = (
        100 * users_gt_5 / len(reposts_per_user)
        if reposts_per_user else 0
    )

    # -------------------------
    # Repost timestamps
    # -------------------------
    total_reposts = 0
    reposts_with_time = 0

    for u in users.values():
        for r in u.get("reposts_of_posts", []):
            total_reposts += 1
            if r.get("reposted_at"):
                reposts_with_time += 1

    pct_reposts_with_time = (
        100 * reposts_with_time / total_reposts
        if total_reposts else 0
    )

    # -------------------------
    # Print results
    # -------------------------
    print("===== BASIC DATASET STATS =====")
    print(f"Users: {len(users)}")
    print(f"Posts: {len(posts_dict)}")
    print()

    print(
        f"Authors followed per user → "
        f"avg: {avg_authors_followed:.2f}, "
        f"max: {max_authors_followed}"
    )

    print(
        f"Reposts per post → "
        f"avg: {avg_reposts_per_post:.2f}, "
        f"max: {max_reposts_per_post}"
    )

    print(
        f"Posts with >5 reposts → "
        f"{posts_gt_5} "
        f"({pct_posts_gt_5:.2f}%)"
    )

    print(
        f"Users who reposted >5 posts → "
        f"{users_gt_5} "
        f"({pct_users_gt_5:.2f}%)"
    )

    print(
        f"Reposts with timestamp → "
        f"{reposts_with_time}/{total_reposts} "
        f"({pct_reposts_with_time:.2f}%)"
    )


In [ ]:
def print_posts_with_more_than_x_reposts(posts_dict,x):
    count = sum(
        p.get("repostCount", 0) > x
        for p in posts_dict.values()
    )
    print(f"Posts with >{x} reposts: {count}/{len(posts_dict)}")


print_posts_with_more_than_x_reposts(post_dict,0)

Posts with >0 reposts: 6230/19294


In [ ]:


print_basic_stats(users,post_dict)

===== BASIC DATASET STATS =====
Users: 9914
Posts: 19294

Authors followed per user → avg: 1.67, max: 13
Reposts per post → avg: 1.68, max: 100
Posts with >5 reposts → 1106 (5.73%)
Users who reposted >5 posts → 4 (0.04%)
Reposts with timestamp → 2242/2242 (100.00%)
